# Helper functions

In [ ]:
# Import
%load_ext autoreload
%autoreload 2

import json
import os
import pandas as pd
import re
import sys
from pathlib import Path

# Add SynFlow to path in order to import modules
repo_root = "../"
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
from SynFlow.SCD import print_jsd_by_period, plot_jsd_by_period, plot_items_jsd_by_period

# Analysis

In [ ]:
target_lemma_POS = 'kokärt'
target_lemma = target_lemma_POS.split('_')[0]
target_pos = 'TAR'
keyword_string = f'{target_lemma}\t{target_pos}' # Or you can use the full POS for precision (e.g., {target_lemma}\tNOUN)
period = '1-2'
corpus_path = f'../SemEval_swe_SemEval/merged_corpus/'
fname_pattern = re.compile(
    rf'swe(?P<id>\d+)_reparsed.txt$'
)

In [ ]:
output_folder = Path(f'../case_studies/SemEval_swe_SemEval')
input_SCD = output_folder / 'input' / 'SCD' /f'{target_lemma}-{target_pos}-{period}'
os.makedirs(input_SCD, exist_ok=True)

In [ ]:
fname_df_pattern = re.compile(
    rf'{target_lemma}/swe'   # fixed prefix
    rf'(?P<id>\d+)_'                            # capture numeric ID
    rf'reparsed'                           # capture year
    rf'.txt/\d+$'             #  and extension and line number
)

all_sfillers_df_path = f'../case_studies/SemEval_swe_SemEval/output/{target_lemma}-{target_pos}-{period}/Explorer/{target_lemma}_samples_sfillerdf_all_no_POS.csv'

### uppläggning

### chi_amod

In [ ]:
upplaggning_chiamod_df_path = f'{input_SCD}/{target_lemma}_chi_amod_1-2.csv'
print(upplaggning_chiamod_df_path)
interest_slot = 'chi_amod'
# Extract the specific slot-filler column
from SynFlow.Explorer.sfiller_df import extract_1_slot_col
upplaggning_chiamod_df = extract_1_slot_col(all_sfillers_df_path, interest_slot, output_path=upplaggning_chiamod_df_path)
# Filter by minimum frequency. Only use this with df that has 1 slot column
from SynFlow.Explorer.sfiller_df import filter_frequency_sfiller
filter_frequency_sfiller(upplaggning_chiamod_df_path, interest_slot, 2)

### chi_nmod

In [ ]:
upplaggning_chinmod_df_path = f'{input_SCD}/{target_lemma}_chi_nmod_1-2.csv'
print(upplaggning_chinmod_df_path)
interest_slot = 'pa_obl'
# Extract the specific slot-filler column
from SynFlow.Explorer.sfiller_df import extract_1_slot_col
upplaggning_chinmod_df = extract_1_slot_col(all_sfillers_df_path, interest_slot, output_path=upplaggning_chinmod_df_path)
# Filter by minimum frequency. Only use this with df that has 1 slot column
from SynFlow.Explorer.sfiller_df import filter_frequency_sfiller
filter_frequency_sfiller(upplaggning_chinmod_df_path, interest_slot, 2)

#### Frequency Changes

In [ ]:
from SynFlow.SCD.freq import plot_freq_top_union_sfillers_by_period
slot_type = 'chi_amod'
slot_df_path = upplaggning_chiamod_df_path

# Bar chart (absolute freq)
plot_freq_top_union_sfillers_by_period(slot_df_path, 
                              slot_type=slot_type,
                              top_n=10,
                              normalized=True, # Normalised or Raw count
                              time_col='subfolder',
                              )

#### Slot Filler JSD

In [ ]:
from SynFlow.SCD.jsd import sfillers_jsd_by_period
sfillers_js_df = pd.read_csv(slot_df_path)
sfillers_js_results = sfillers_jsd_by_period(sfillers_js_df, word_col=slot_type, period_col='subfolder', top_n=10)
print_jsd_by_period(sfillers_js_results)
plot_jsd_by_period(sfillers_js_results)
plot_items_jsd_by_period(sfillers_js_results, top_n=15, cols=3)